# OneVoice V2 — Local runtime artifacts

Chuẩn bị các model đã đạt benchmark vào Google Drive và tạo `runtime_demo_local.yaml`. Notebook không fine-tune, không dùng INT8 SenseVoice failed và không thay đổi `config/config.yaml` trong Git.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
REPO = Path('/content/OneVoice')
ROOT = Path('/content/drive/MyDrive/OneVoice')
MODEL_ROOT = ROOT / 'models'
CACHE_ROOT = ROOT / 'model_cache'

if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, str(REPO / 'src'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'huggingface_hub', 'sherpa-onnx', 'funasr-onnx', 'PyYAML', 'soundfile', 'numpy==2.2.6', 'transformers==4.57.1', 'tokenizers==0.22.1', 'sentencepiece==0.2.0'], check=True)


In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
from asr.asr_manager import GIPFORMER_INT8_FILES, GIPFORMER_REPO, GIPFORMER_REVISION

MT_VI2EN = MODEL_ROOT / 'envit5_finetuned_vi2en_v1'
MT_EN2VI = MODEL_ROOT / 'envit5_finetuned_en2vi_v1/best'
GIPFORMER = MODEL_ROOT / 'gipformer'
SENSEVOICE = MODEL_ROOT / 'sensevoice_en_construction_v1_onnx_fp32'

if not (MT_VI2EN / 'config.json').is_file():
    snapshot_download('platypus123/onevoice-envit5-vi-en', local_dir=str(MT_VI2EN))
else:
    print('VI→EN MT already staged:', MT_VI2EN)

GIPFORMER.mkdir(parents=True, exist_ok=True)
for filename in GIPFORMER_INT8_FILES.values():
    target = GIPFORMER / filename
    if target.is_file():
        print('GIPFormer artifact already staged:', filename)
    else:
        hf_hub_download(GIPFORMER_REPO, filename, revision=GIPFORMER_REVISION, local_dir=str(GIPFORMER))

for path, marker in ((MT_EN2VI, 'config.json'), (SENSEVOICE, 'model.onnx')):
    if not (path / marker).is_file():
        raise FileNotFoundError(f'Missing verified Drive model: {path} ({marker})')
print('All required model directories are present.')


In [ ]:
REVIEWED_SAFETY_CSV = ROOT / 'review/safety_fast_path_review.csv'
SAFETY_MANIFEST = ROOT / 'artifacts/safety_audio_v1/manifest.json'
RUNTIME_CONFIG = ROOT / 'configs/runtime_demo_local.yaml'

command = [
    sys.executable, 'scripts/write_runtime_override.py',
    '--output', str(RUNTIME_CONFIG),
    '--gipformer-dir', str(GIPFORMER),
    '--sensevoice-dir', str(SENSEVOICE),
    '--mt-vi2en-dir', str(MT_VI2EN),
    '--mt-en2vi-dir', str(MT_EN2VI),
    '--safety-csv', str(REVIEWED_SAFETY_CSV),
    '--safety-manifest', str(SAFETY_MANIFEST),
    '--profile', 'development',
]
subprocess.run(command, check=True)
print(RUNTIME_CONFIG.read_text(encoding='utf-8'))


In [ ]:
import json
sys.path.insert(0, str(REPO / 'src'))
from safety.audio_store import SafetyAudioStore

store = SafetyAudioStore(SAFETY_MANIFEST, source_csv=REVIEWED_SAFETY_CSV)
audio, sample_rate = store.get('SAFE2_0001', 'vi2en')
assert audio.size > 0 and sample_rate > 0
print(f'Runtime artifact config ready: {RUNTIME_CONFIG}')
print(f'Safety fast-path smoke: SAFE2_0001/vi2en = {len(audio)} samples at {sample_rate} Hz')
